In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 14 — Ejercicio 1
# ---------------------------------------------------------------

import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# --- Nuevas features -------------------------------------------
df_all["TotalBath"] = (
    df_all["BsmtFullBath"].fillna(0)
    + df_all["BsmtHalfBath"].fillna(0)  * 0.5
    + df_all["FullBath"].fillna(0)
    + df_all["HalfBath"].fillna(0)      * 0.5
)

df_all["HasPool"] = (df_all["PoolArea"] > 0).astype(int)

df_all["PorchSF"] = (
    df_all["OpenPorchSF"].fillna(0)
    + df_all["EnclosedPorch"].fillna(0)
    + df_all["3SsnPorch"].fillna(0)
    + df_all["ScreenPorch"].fillna(0)
)

# --- Recrear el preprocesador y pipeline con las nuevas features ---
# Redefinir las listas de columnas numéricas y categóricas del df_all actualizado
cols_num_updated = df_all.select_dtypes(include=[np.number]).columns.tolist()
cols_cat_updated = df_all.select_dtypes(include=['object']).columns.tolist()

# Asegurarse de que 'Id' no sea tratado como una feature numérica
if 'Id' in cols_num_updated:
    cols_num_updated.remove('Id')

# Re-crear pipelines numérico y categórico
pipe_num_ext = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler()),
])

pipe_cat_ext = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
])

# Re-crear el preprocesador con las columnas actualizadas
preprocesador_ext = ColumnTransformer([
    ('num', pipe_num_ext, cols_num_updated),
    ('cat', pipe_cat_ext, cols_cat_updated),
])

# Extraer los mejores parámetros de XGBoost de la búsqueda aleatoria anterior
best_xgb_params = {k.replace('clf__', ''): v for k, v in rs.best_params_.items()}

# Re-crear el pipeline de XGBoost con el nuevo preprocesador y los mejores parámetros
xgb_pipe_ext = Pipeline([
    ('prep', preprocesador_ext),
    ('clf', XGBRegressor(
        **best_xgb_params,
        random_state=semilla,
        n_jobs=-1,
        verbosity=0,
    )),
])

# Preparar X_train para la evaluación (usa el df_all modificado)
X_train_for_eval = df_all.iloc[:n_train].copy()

# Evaluar el pipeline de XGBoost extendido con validación cruzada
# y_train ya contiene el log1p(SalePrice) del df_train original
scores_ext = cross_val_score(
    xgb_pipe_ext, X_train_for_eval, y_train,
    cv=kf, # Usar kf (KFold) definido anteriormente
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
print(f"RMSE CV con nuevas features: {(-scores_ext).mean():.5f}")
print(f"RMSE baseline (tuneado): 0.1276")
print(f"RMSE con nuevas features: {(-scores_ext).mean():.5f}")
print(f"📉 Mejora: {0.1276 - (-scores_ext).mean():.5f}")

In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 14 — Ejercicio 2
# ---------------------------------------------------------------

try:
    import lightgbm as lgb
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'lightgbm', '--quiet'])
    import lightgbm as lgb
print(f"lightgbm: {lgb.__version__}")

from sklearn.pipeline import Pipeline
import time

# Pipeline idéntico al del capítulo pero con LightGBM
pipeline_lgbm = Pipeline([
    ("prep",  preprocesador),    # mismo que pipeline_xgb
    ("model", lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )),
])

t0 = time.time()
scores_lgbm = cross_val_score(
    pipeline_lgbm, X_train_raw, y_train,
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
)
print(f"RMSE CV LightGBM: {(-scores_lgbm).mean():.5f}", f"| Tiempo: {time.time()-t0:.1f}s")

In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 14 — Ejercicio 3
# ---------------------------------------------------------------

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import numpy as np, pandas as pd

# Define X_train and y_train_log
X_train = X_train_raw
y_train_log = y_train
X_test = X_test_raw
X_test_ids = df_test['Id']

# Pipelines individuales (preprocesador ya ajustado en el capítulo)
ridge_pipe = Pipeline([("prep", preprocesador), ("clf", Ridge(alpha=10.0))])
rf_pipe    = Pipeline([("prep", preprocesador), ("clf", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))])
xgb_pipe   = Pipeline([("prep", preprocesador), ("clf", XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42, eval_metric="rmse", verbosity=0))])

modelos_base = [
    ("ridge", ridge_pipe),
    ("rf",    rf_pipe),
    ("xgb",   xgb_pipe),
]

stacking_reg = StackingRegressor(
    estimators=modelos_base,
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1
)

scores_stk = cross_val_score(
    stacking_reg, X_train, y_train_log,
    cv=5, scoring='neg_root_mean_squared_error'
)
print(f"RMSE CV Stacking: {(-scores_stk).mean():.5f}")

# Entrenar y generar submission
stacking_reg.fit(X_train, y_train_log)
y_test_pred_log = stacking_reg.predict(X_test)
y_test_pred     = np.expm1(y_test_pred_log)

submission = pd.DataFrame({
    "Id":        X_test_ids,   # IDs del conjunto de test de Kaggle
    "SalePrice": y_test_pred,
})
submission.to_csv("submission_stacking.csv", index=False)
print("submission_stacking.csv generado")